# Example: SIM Parameter Estimation, Bootstrap Uncertainty, and the Maximum Sharpe Ratio Portfolio

In this example, we load the frozen synthetic training dataset (20 years of JumpHMM-generated market data), estimate Single Index Model parameters for a portfolio of assets via regularized OLS, bootstrap the sampling distribution to quantify parameter uncertainty, construct the SIM-derived covariance matrix, and solve for the tangency (maximum Sharpe ratio) portfolio. Let's dive in!

> __Learning Objectives:__
>
> * __SIM parameter estimation and bootstrap uncertainty:__ Estimate alpha, beta, and residual volatility for each asset via regularized OLS regression on synthetic training data. Bootstrap the sampling distribution to quantify uncertainty with 95% confidence intervals.
> * __SIM covariance vs. sample covariance:__ Construct the SIM-derived covariance matrix and compare its eigenstructure, condition number, and stability to the naive sample covariance. Understand why the SIM covariance is better conditioned for portfolio optimization.
> * __Maximum Sharpe ratio portfolio:__ Solve the second-order cone program to find the tangency portfolio and plot the Capital Market Line on the efficient frontier. Compare tangency, min-variance, and equal-weight portfolios in a scorecard.

## Setup, Data and Prerequisites
We begin by loading the `eCornellAIFinance` package and the frozen synthetic training dataset. The cell below loads all required packages and helper functions.

In [ ]:
# --- Load packages and helper functions ---
include("Include.jl");

Load the 20-year synthetic training dataset using the [the `MySyntheticTrainingDataSet()` function](../../code/docs/build/session1.html) and select a subset of 10 tickers spanning different sectors and risk profiles. The code below computes daily log growth rates for the market index and each ticker, storing the per-ticker results in the `ticker_returns::Dict{String, Vector{Float64}}` variable.

In [ ]:
# --- Step 1: Load the frozen synthetic training dataset ---
training_data = MySyntheticTrainingDataSet();
dataset = training_data["dataset"];           # Dict of ticker DataFrames
market_prices = training_data["market_prices"]; # Vector of market index prices
T_total = training_data["n_days"];             # total number of trading days
Δt = 1.0 / 252.0;                             # daily time step (fraction of year)

# --- Step 2: Select a portfolio of tickers ---
my_tickers = ["AAPL", "MSFT", "NVDA", "JNJ", "JPM", "PG", "XOM", "BA", "GS", "AMD"];
K = length(my_tickers);

# --- Step 3: Compute market log growth rates ---
market_returns = [(1.0/Δt) * log(market_prices[t]/market_prices[t-1]) for t ∈ 2:T_total];
T = length(market_returns);

# --- Step 4: Compute per-ticker log growth rates ---
ticker_returns = Dict{String, Vector{Float64}}();
for ticker ∈ my_tickers
    prices = dataset[ticker][:, :close];  # extract closing prices for this ticker
    g = [(1.0/Δt) * log(prices[t]/prices[t-1]) for t ∈ 2:T_total];
    ticker_returns[ticker] = g;
end

# --- Step 5: Print summary ---
println("Training data: $(T) daily returns × $(K) tickers")
println("Market CAGR: $(round((market_prices[end]/market_prices[1])^(1.0/20) - 1, digits=3)*100)%")
println("Selected tickers: $(my_tickers)")

### Implementation
We define two helper functions used later in the notebook. The `plot_sim_fit(ticker)` function generates a scatter plot of SIM-predicted vs. actual returns for a given ticker, using the bootstrap results and ticker return data. The `scorecard_row(name, rets)` function computes annualized return, volatility, Sharpe ratio, and maximum drawdown for a portfolio return series, returning a formatted tuple for display.

In [ ]:
"""
    plot_sim_fit(ticker::String) -> Plots.Plot

Generate a scatter plot of SIM-predicted vs. actual daily log growth rates for `ticker`.
Uses `bootstrap_results[ticker]` for the point estimate and `ticker_returns[ticker]` for actual returns.
Returns a `Plots.Plot` object with predicted returns on the x-axis, actual returns on the y-axis, 
and a dashed x = y reference line.
"""
function plot_sim_fit(ticker)
    est = bootstrap_results[ticker]["point_estimate"];
    y = ticker_returns[ticker];                         # actual returns
    ŷ = est.α .+ est.β .* market_returns;              # SIM-predicted returns
    lims = (min(minimum(ŷ), minimum(y)), max(maximum(ŷ), maximum(y)));
    scatter(ŷ, y, alpha=0.3, ms=2, color=:navy, label="",
        xlabel="Predicted gᵢ", ylabel="Actual gᵢ",
        title="$(ticker) (R²=$(round(est.r², digits=3)), β=$(round(est.β, digits=2)))");
    plot!([lims...], [lims...], lw=2, ls=:dash, color=:red, label="x = y")
end

"""
    scorecard_row(name::String, rets::Vector{Float64}) -> Tuple

Compute annualized return (%), volatility (%), Sharpe ratio, and maximum drawdown (%) 
for a daily portfolio return series `rets`. Returns a tuple 
`(name, return_pct, vol_pct, sharpe, maxdd_pct)` with values rounded for display.
Uses `compute_drawdown(rets)` to calculate the maximum drawdown.
"""
function scorecard_row(name, rets)
    ret_ann = mean(rets) * 252 * 100;              # annualized return (%)
    vol_ann = std(rets) * sqrt(252) * 100;          # annualized volatility (%)
    sr = ret_ann / vol_ann;                          # Sharpe ratio
    dd = compute_drawdown(rets) * 100;               # max drawdown (%)
    return (name, round(ret_ann, digits=2), round(vol_ann, digits=2),
            round(sr, digits=3), round(dd, digits=2))
end;

___
## Task 1: Estimate SIM Parameters and Bootstrap the Sampling Distribution
We estimate the Single Index Model parameters $(\alpha_i, \beta_i, \sigma_{\varepsilon,i})$ for each ticker via regularized OLS regression, then bootstrap the sampling distribution (1,000 replicates) to quantify uncertainty.

> __What are we going to do?__
>
> For each of the $K$ tickers, we regress asset returns on market returns using [the `estimate_sim(...)` function](../../code/docs/build/session1.html), then run [the `bootstrap_sim(...)` function](../../code/docs/build/session1.html) with 1,000 replicates to get 95% confidence intervals for $\alpha$ and $\beta$. We compare the bootstrap CIs to the theoretical standard errors.

High-beta tickers (NVDA, AMD) will have $\beta > 1$ with tight confidence intervals (high $R^{2}$), while defensive tickers (JNJ, PG) will have $\beta < 1$. The bootstrap CIs should closely match the theoretical CIs, confirming the Gaussian error assumption is reasonable.

The code below estimates SIM parameters for each ticker and stores the results in the `sim_estimates::Vector{MySIMParameterEstimate}` variable.

In [ ]:
# --- Step 1: Estimate SIM parameters for each ticker ---
sim_estimates = MySIMParameterEstimate[];
for ticker ∈ my_tickers
    est = estimate_sim(market_returns, ticker_returns[ticker], ticker; δ=0.0, Δt=Δt);
    push!(sim_estimates, est);
end

# --- Step 2: Build a summary DataFrame ---
params_df = DataFrame(
    "Ticker" => [e.ticker for e ∈ sim_estimates],
    "α (ann bps)" => [round(e.α * 252 * 10000, digits=1) for e ∈ sim_estimates],
    "β" => [round(e.β, digits=3) for e ∈ sim_estimates],
    "σ_ε (ann %)" => [round(e.σ_ε * sqrt(252) * 100, digits=1) for e ∈ sim_estimates],
    "R²" => [round(e.r², digits=3) for e ∈ sim_estimates]
);

# --- Step 3: Display results ---
println("SIM Parameter Estimates:")
println("═"^60)
pretty_table(params_df, tf = tf_markdown)

**Bootstrap Deep Dive:** Let's run the full bootstrap for one ticker (NVDA) and compare the empirical sampling distribution to the theoretical prediction.

> __What should you see?__
>
> The histograms of bootstrap $\hat{\alpha}$ and $\hat{\beta}$ should be approximately normal. The bootstrap mean should match the point estimate, and the bootstrap standard deviation should match the theoretical standard error.

The code below runs [the `bootstrap_sim(...)` function](../../code/docs/build/session1.html) for NVDA with 1,000 replicates and prints the comparison.

In [ ]:
# --- Step 1: Bootstrap NVDA with 1,000 replicates ---
bs_nvda = bootstrap_sim(market_returns, ticker_returns["NVDA"], "NVDA";
    δ=0.0, Δt=Δt, n_bootstrap=1000, seed=42);

# --- Step 2: Extract point estimate and print comparison ---
est_nvda = bs_nvda["point_estimate"];
println("NVDA Bootstrap Results (1,000 replicates):")
println("═"^55)
println("  α: point = $(round(est_nvda.α, digits=6)), bootstrap mean = $(round(bs_nvda["alpha_mean"], digits=6))")
println("  β: point = $(round(est_nvda.β, digits=4)), bootstrap mean = $(round(bs_nvda["beta_mean"], digits=4))")
println()
println("  α 95% CI: ($(round(bs_nvda["alpha_ci_95"][1], digits=6)), $(round(bs_nvda["alpha_ci_95"][2], digits=6)))")
println("  β 95% CI: ($(round(bs_nvda["beta_ci_95"][1], digits=4)), $(round(bs_nvda["beta_ci_95"][2], digits=4)))")
println()

# --- Step 3: Compare bootstrap SE to theoretical SE ---
println("  Bootstrap SE(β): $(round(bs_nvda["beta_std"], digits=5))")
println("  Theoretical SE(β): $(round(bs_nvda["theoretical_se"][2], digits=5))")
println("  Ratio: $(round(bs_nvda["beta_std"] / bs_nvda["theoretical_se"][2], digits=3))")

The `let...end` block below plots histograms of bootstrap $\hat{\alpha}$ and $\hat{\beta}$ for NVDA, with the point estimate and 95% CI marked.

In [ ]:
let
    # --- Left panel: bootstrap distribution of α̂ ---
    p1 = histogram(bs_nvda["alpha_samples"], bins=50, alpha=0.7, color=:steelblue,
        xlabel="α̂", ylabel="Count", title="Bootstrap Distribution of α̂ (NVDA)", label="");
    vline!(p1, [est_nvda.α], lw=2, color=:red, label="Point estimate");
    vline!(p1, [bs_nvda["alpha_ci_95"]...], lw=1.5, ls=:dash, color=:orange, label="95% CI");

    # --- Right panel: bootstrap distribution of β̂ ---
    p2 = histogram(bs_nvda["beta_samples"], bins=50, alpha=0.7, color=:steelblue,
        xlabel="β̂", ylabel="Count", title="Bootstrap Distribution of β̂ (NVDA)", label="");
    vline!(p2, [est_nvda.β], lw=2, color=:red, label="Point estimate");
    vline!(p2, [bs_nvda["beta_ci_95"]...], lw=1.5, ls=:dash, color=:orange, label="95% CI");

    # --- Combine panels ---
    plot(p1, p2, layout=(1,2), size=(1000, 400), legend=:topright)
end

**All Tickers:** Bootstrap all 10 tickers and display the comparison table (point estimates, bootstrap 95% CIs, and theoretical SEs). The results are stored in the `bootstrap_results::Dict{String, Dict{String,Any}}` variable.

In [ ]:
# --- Step 1: Bootstrap all tickers (1,000 replicates each) ---
bootstrap_results = Dict{String, Dict{String,Any}}();
for ticker ∈ my_tickers
    bootstrap_results[ticker] = bootstrap_sim(market_returns, ticker_returns[ticker], ticker;
        δ=0.0, Δt=Δt, n_bootstrap=1000, seed=42);
end

# --- Step 2: Build comparison table ---
comparison = DataFrame(
    "Ticker" => String[], "β̂" => Float64[], "β 95% CI" => String[],
    "Boot SE(β)" => Float64[], "Theory SE(β)" => Float64[], "R²" => Float64[]
);

for ticker ∈ my_tickers
    bs = bootstrap_results[ticker];
    est = bs["point_estimate"];
    ci = bs["beta_ci_95"];
    push!(comparison, (ticker, round(est.β, digits=3),
        "[$(round(ci[1], digits=3)), $(round(ci[2], digits=3))]",
        round(bs["beta_std"], digits=4), round(bs["theoretical_se"][2], digits=4),
        round(est.r², digits=3)));
end

# --- Step 3: Display results ---
println("Bootstrap Comparison, All Tickers:")
println("═"^70)
pretty_table(comparison, tf = tf_markdown)

**Visualize:** Predicted vs. actual returns for a high-$R^{2}$ ticker (NVDA) and a low-$R^{2}$ ticker (JNJ).

> __What should you see?__
>
> NVDA should cluster tightly around the $x=y$ line (high $R^{2}$). JNJ should show more scatter, because its returns are less explained by the market factor alone.

The `let...end` block below generates the scatter plots for both tickers.

In [ ]:
let
    # --- Generate side-by-side panels ---
    p1 = plot_sim_fit("NVDA");
    p2 = plot_sim_fit("JNJ");
    plot(p1, p2, layout=(1,2), size=(1000, 450))
end

___
## Task 2: Build the SIM Covariance Matrix and Compare to Sample Covariance
We construct the SIM-derived covariance matrix $\boldsymbol{\Sigma}^{\text{SIM}}$ and compare its properties to the naive sample covariance $\hat{\boldsymbol{\Sigma}}$.

> __What are we going to do?__
>
> Build $\boldsymbol{\Sigma}^{\text{SIM}}$ from the estimated $\beta_i$ and $\sigma_{\varepsilon,i}$ values using [the `build_sim_covariance(...)` function](../../code/docs/build/session1.html), compute the sample covariance for comparison, and examine eigenvalue spectra, condition numbers, and correlation heatmaps.

The SIM covariance should have a lower condition number (better conditioning for optimization) and a cleaner eigenvalue spectrum (one dominant market factor plus idiosyncratic noise). The code below builds both covariance matrices and prints the condition number comparison.

In [ ]:
# --- Step 1: Compute annualized market volatility ---
σ_m = std(market_returns);
println("Market volatility (annualized): $(round(σ_m * sqrt(252) * 100, digits=1))%")

# --- Step 2: Build the SIM covariance matrix ---
Σ_sim = build_sim_covariance(sim_estimates, σ_m; Δt=Δt);

# --- Step 3: Build the sample covariance matrix ---
R_matrix = hcat([ticker_returns[t] for t ∈ my_tickers]...);  # T × K return matrix
Σ_sample = cov(R_matrix);

# --- Step 4: Verify SIM covariance properties ---
@assert issymmetric(Σ_sim) "SIM covariance must be symmetric"
@assert isposdef(Σ_sim) "SIM covariance must be positive definite"

# --- Step 5: Compare condition numbers ---
κ_sim = cond(Σ_sim);
κ_sample = cond(Σ_sample);
println("\nCondition numbers:")
println("  Sample covariance: $(round(κ_sample, digits=1))")
println("  SIM covariance:    $(round(κ_sim, digits=1))")
println("  Improvement: $(round(κ_sample / κ_sim, digits=1))×")

The `let...end` block below generates side-by-side correlation heatmaps and eigenvalue spectra for the sample and SIM covariance matrices.

In [ ]:
let
    # --- Step 1: Convert covariance matrices to correlation matrices ---
    D_sim = diagm(1.0 ./ sqrt.(diag(Σ_sim)));
    C_sim = D_sim * Σ_sim * D_sim;
    D_samp = diagm(1.0 ./ sqrt.(diag(Σ_sample)));
    C_sample = D_samp * Σ_sample * D_samp;

    # --- Step 2: Heatmap of sample correlation ---
    p1 = heatmap(C_sample, title="Sample Correlation", xticks=(1:K, my_tickers),
        yticks=(1:K, my_tickers), clims=(-1,1), color=:RdBu, xrotation=45);

    # --- Step 3: Heatmap of SIM correlation ---
    p2 = heatmap(C_sim, title="SIM Correlation", xticks=(1:K, my_tickers),
        yticks=(1:K, my_tickers), clims=(-1,1), color=:RdBu, xrotation=45);

    # --- Step 4: Eigenvalue spectrum comparison ---
    λ_sample = eigvals(Σ_sample) |> sort |> reverse;
    λ_sim = eigvals(Σ_sim) |> sort |> reverse;
    p3 = plot(1:K, λ_sample .* 10000, marker=:circle, label="Sample", lw=2,
        xlabel="Component", ylabel="Eigenvalue (×10⁴)", title="Eigenvalue Spectrum");
    plot!(p3, 1:K, λ_sim .* 10000, marker=:diamond, label="SIM", lw=2, ls=:dash);

    # --- Step 5: Combine into layout ---
    plot(p1, p2, p3, layout=@layout([a b; c{0.5h}]), size=(900, 700))
end

___
## Task 3: Solve for the Maximum Sharpe Ratio Portfolio
We solve the SOCP to find the tangency portfolio (the allocation that maximizes the Sharpe ratio) and compare it to the min-variance portfolio and an equal-weight benchmark.

> __What are we going to do?__
>
> Build the Sharpe ratio problem from SIM parameters using [the `build(MySharpeRatioPortfolioChoiceProblem, ...)` function](../../code/docs/build/session1.html), solve via [the `solve_max_sharpe(...)` function](../../code/docs/build/session1.html), compute the efficient frontier, plot the Capital Market Line through the tangency portfolio, and produce a comparison scorecard.

The tangency portfolio will have higher return and higher volatility than the min-variance portfolio, but a better Sharpe ratio. It will also be differently concentrated, loading more on high-alpha tickers rather than just low-variance ones. The code below builds both the Sharpe ratio problem and the min-variance problem, then displays the weight comparison.

In [ ]:
# --- Step 1: Extract SIM parameters for the Sharpe ratio problem ---
α_vec = [e.α for e ∈ sim_estimates];   # alpha vector (K × 1)
β_vec = [e.β for e ∈ sim_estimates];   # beta vector (K × 1)
gm_mean = mean(market_returns);         # mean market return
rf = 0.05 / 252;                        # daily risk-free rate (5% annualized)

# --- Step 2: Build and solve the maximum Sharpe ratio problem (SOCP) ---
sharpe_problem = build(MySharpeRatioPortfolioChoiceProblem, (
    Σ = Σ_sim, risk_free_rate = rf, α = α_vec, β = β_vec,
    gₘ = gm_mean, bounds = hcat(zeros(K), ones(K))));
sharpe_result = solve_max_sharpe(sharpe_problem);

# --- Step 3: Build and solve the min-variance problem for comparison ---
μ_hat = vec(mean(R_matrix, dims=1));  # sample mean return vector
minvar_problem = build(MyPortfolioAllocationProblem;
    μ = μ_hat, Σ = Σ_sim, bounds = hcat(zeros(K), ones(K)), R = rf);
minvar_result = solve_minvariance(minvar_problem);

# --- Step 4: Equal-weight benchmark ---
w_equal = fill(1.0/K, K);

# --- Step 5: Display weight comparison ---
weights_df = DataFrame("Ticker" => my_tickers,
    "Tangency (%)" => round.(sharpe_result["weights"] .* 100, digits=1),
    "Min-Var (%)" => round.(minvar_result.weights .* 100, digits=1),
    "Equal (%)" => round.(w_equal .* 100, digits=1));

println("Portfolio Weights Comparison:")
println("═"^55)
pretty_table(weights_df, tf = tf_markdown)
println("\nSharpe Ratio (tangency): $(round(sharpe_result["sharpe_ratio"], digits=3))")

The `let...end` block below generates a grouped bar chart comparing portfolio weights across the tangency, min-variance, and equal-weight portfolios.

In [ ]:
let
    # --- Grouped bar chart: tangency vs min-variance vs equal-weight ---
    groupedbar(my_tickers,
        hcat(sharpe_result["weights"] .* 100, minvar_result.weights .* 100, w_equal .* 100),
        bar_position=:dodge, bar_width=0.25,
        label=["Tangency" "Min-Variance" "Equal-Weight"],
        color=[:coral :steelblue :gray60],
        ylabel="Weight (%)", xlabel="Ticker",
        title="Portfolio Weights: Tangency vs Min-Var vs Equal",
        size=(800, 450), legend=:topright)
end

**Efficient Frontier and Capital Market Line:** We sweep the target return to trace the efficient frontier, then plot the tangency portfolio and the CML.

> __What should you see?__
>
> The CML is a straight line from the risk-free rate through the tangency portfolio (red star). Every point on the CML dominates the corresponding point on the frontier at the same risk level.

The `let...end` block below sweeps target returns to compute the frontier using [the `solve_minvariance(...)` function](../../code/docs/build/session1.html), then overlays the CML and portfolio markers.

In [ ]:
let
    # --- Step 1: Sweep target returns to trace the efficient frontier ---
    R_sweep = range(0.0, stop=maximum(μ_hat)*0.95, length=100) |> collect;
    frontier_risk = Float64[];
    frontier_return = Float64[];
    for R_i ∈ R_sweep
        try
            prob = build(MyPortfolioAllocationProblem;
                μ = μ_hat, Σ = Σ_sim, bounds = hcat(zeros(K), ones(K)), R = R_i);
            sol = solve_minvariance(prob);
            push!(frontier_risk, sqrt(sol.variance) * sqrt(252) * 100);   # annualized vol (%)
            push!(frontier_return, sol.expected_return * 252 * 100);       # annualized return (%)
        catch; end
    end

    # --- Step 2: Compute annualized metrics for each portfolio ---
    w_t = sharpe_result["weights"];
    tang_ret = dot(μ_hat, w_t) * 252 * 100;
    tang_vol = sqrt(dot(w_t, Σ_sim * w_t)) * sqrt(252) * 100;
    rf_ann = rf * 252 * 100;
    mv_ret = minvar_result.expected_return * 252 * 100;
    mv_vol = sqrt(minvar_result.variance) * sqrt(252) * 100;
    eq_ret = dot(μ_hat, w_equal) * 252 * 100;
    eq_vol = sqrt(dot(w_equal, Σ_sim * w_equal)) * sqrt(252) * 100;

    # --- Step 3: Compute the Capital Market Line ---
    cml_x = range(0, stop=tang_vol*1.5, length=50);
    cml_slope = (tang_ret - rf_ann) / tang_vol;
    cml_y = rf_ann .+ cml_slope .* cml_x;

    # --- Step 4: Plot frontier, CML, and portfolio markers ---
    p = plot(frontier_risk, frontier_return, lw=2, color=:steelblue, label="Efficient Frontier",
        xlabel="Volatility (annual %)", ylabel="Expected Return (annual %)",
        title="Efficient Frontier, Tangency Portfolio, and CML", legend=:topleft, size=(750, 500));
    plot!(p, collect(cml_x), collect(cml_y), lw=2, ls=:dash, color=:darkred, label="Capital Market Line");
    scatter!(p, [tang_vol], [tang_ret], marker=:star5, ms=14, color=:red, label="Tangency (max SR)");
    scatter!(p, [mv_vol], [mv_ret], marker=:circle, ms=8, color=:steelblue, label="Min-Variance");
    scatter!(p, [eq_vol], [eq_ret], marker=:diamond, ms=8, color=:green, label="Equal-Weight");
    scatter!(p, [0], [rf_ann], marker=:square, ms=6, color=:black, label="Risk-Free");
    p
end

**Scorecard:** Compare the three portfolios across key metrics.

> __What should you see?__
>
> The tangency portfolio should have the highest Sharpe ratio but also the highest volatility. The min-variance portfolio should have the lowest volatility but a lower Sharpe. Equal-weight sits between them.

The `let...end` block below computes annualized return, volatility, Sharpe ratio, and maximum drawdown for each portfolio using [the `compute_drawdown(...)` function](../../code/docs/build/session1.html).

In [ ]:
let
    # --- Step 1: Compute daily portfolio returns for each strategy ---
    R = R_matrix;
    w_t = sharpe_result["weights"];
    w_mv = minvar_result.weights;
    tang_rets = R * w_t;      # tangency portfolio daily returns
    mv_rets = R * w_mv;       # min-variance portfolio daily returns
    eq_rets = R * w_equal;    # equal-weight portfolio daily returns

    # --- Step 2: Build and display scorecard DataFrame ---
    sc = DataFrame([scorecard_row("Tangency", tang_rets),
                    scorecard_row("Min-Variance", mv_rets),
                    scorecard_row("Equal-Weight", eq_rets)],
        ["Portfolio", "Return (%)", "Vol (%)", "Sharpe", "MaxDD (%)"])

    println("═"^60)
    println("  PORTFOLIO SCORECARD")
    println("═"^60)
    pretty_table(sc, tf = tf_markdown)
end

___
## Summary
This example demonstrated end-to-end SIM parameter estimation, bootstrap uncertainty quantification, covariance construction, and maximum Sharpe ratio portfolio optimization on a frozen synthetic dataset.

> __Key Takeaways:__
>
> * **SIM estimation recovers meaningful factor exposures:** High-beta tickers show strong market sensitivity with tight confidence intervals, while defensive tickers show low beta. Bootstrap confidence intervals match theoretical standard errors, confirming the Gaussian error model is reasonable.
> * **SIM covariance is better conditioned than sample covariance:** Fewer parameters reduce estimation noise and produce more stable portfolio weights. This improved conditioning comes at the cost of assuming a single-factor correlation structure.
> * **The tangency portfolio maximizes risk-adjusted return:** It sits at the CML tangent point on the efficient frontier, accepting more variance than the min-variance portfolio. The resulting Sharpe ratio is substantially better than both the min-variance and equal-weight benchmarks.

These techniques provide a foundation for building factor-based portfolio allocation systems that balance estimation stability with return optimization.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.
___